In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


#Ordinale Targets erzeugen (CORN-Style)
def ordinal_targets(y, num_classes):
    return torch.stack(
        [(y >= i).float() for i in range(1, num_classes)],
        dim=1
    )



betw_nn_no = 16  # try with 8
class OrdinalNet(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, betw_nn_no),
            #nn.BatchNorm1d(betw_nn_no),  # <-- Batch Norm (nach Linear oder ReLU)
            nn.ReLU(),
            #nn.Dropout(p=0.2),           # <-- Dropout (20% der Neuronen aus)
            nn.Linear(betw_nn_no, num_classes - 1)
        )
    def forward(self, x):  # forward im aktuellen Modus (hier train)
        return self.net(x)  # logits


        
    
# Deine Daten (aus dem Original-Code)
torch.manual_seed(0)
X = torch.randn(200, 5)
latent = X[:, 0] + 0.5 * X[:, 1]
y = torch.bucketize(latent, boundaries=torch.tensor([-1.0, -0.2, 0.5, 1.2]))
num_classes = 5
y_ord = ordinal_targets(y, num_classes)  # deine Funktion

# 1. DataLoader bauen
dataset = TensorDataset(X, y_ord)  # Paart X und y_ord
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)  # 32er Batches, zufällig gemischt!

# 2. Modell, Optimizer, Criterion (wie vorher)
model = OrdinalNet(5, num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()
epoch_n = 120

# 3. Training Loop mit Mini‑Batches
for epoch in range(1, epoch_n + 1):
    model.train()  # explizit Train-Modus (gut, falls du später Dropout hinzufügst)
    
    epoch_loss = 0.0
    num_batches = 0
    
    # Innere Schleife: Über ALLE Mini-Batches iterieren!
    for batch_X, batch_y_ord in dataloader:
        # 4x pro Epoch: zero_grad, forward, backward, step
        optimizer.zero_grad()
        logits = model(batch_X)  # Jetzt batch_X.shape = (32, 5)!
        loss = criterion(logits, batch_y_ord)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Avg Loss {avg_loss:.4f} (über {num_batches} Batches)")

print("Training fertig!")


Epoch  20 | Avg Loss 0.0950 (über 7 Batches)
Epoch  40 | Avg Loss 0.0466 (über 7 Batches)
Epoch  60 | Avg Loss 0.0293 (über 7 Batches)
Epoch  80 | Avg Loss 0.0210 (über 7 Batches)
Epoch 100 | Avg Loss 0.0158 (über 7 Batches)
Epoch 120 | Avg Loss 0.0136 (über 7 Batches)
Training fertig!


In [4]:
# Vorhersage → ordinale Klasse
model.eval() # 1. Layer-Verhalten auf "Vorhersage" stellen
with torch.no_grad():
    logits = model(X)
    probs = torch.sigmoid(logits)
    y_pred = (probs > 0.5).sum(dim=1)

print("True:", y[:10])
print("Pred:", y_pred[:10])


True: tensor([0, 3, 3, 1, 2, 1, 3, 2, 2, 4])
Pred: tensor([0, 3, 3, 1, 2, 1, 3, 2, 2, 4])
